# IN403 – Unit 9 Lab: LLM Security, Red Teaming, and Guardrails

## Objective
Build a **guardrail layer** for a hospital policy assistant and evaluate it using a red-team prompt set.

## Deliverables
1) `guardrail_rules(prompt)` → **ALLOW / REFUSE / ESCALATE**
2) `guardrail_hybrid(prompt)` using **rules + a Hugging Face classifier**
3) Metrics: classification report, confusion matrix, and **False ALLOW rate**
4) Short write-up: hardest attack category + recommended controls + go/no-go

## Files provided
- `IN403_Unit9_Security_RedTeaming_Dataset.zip`
  - `red_team_prompts.csv`
  - `toy_policies.csv`


In [ ]:
# Optional installs (Colab)
# !pip -q install pandas numpy scikit-learn transformers torch

import os, zipfile, re
import numpy as np
import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix
import torch
from transformers import pipeline

SEED = 42
np.random.seed(SEED)


## 1) Unzip and load dataset


In [ ]:
zip_file = 'IN403_Unit9_Security_RedTeaming_Dataset.zip'
data_dir = 'unit9_data'

if not os.path.exists(data_dir):
    os.makedirs(data_dir, exist_ok=True)
    with zipfile.ZipFile(zip_file, 'r') as z:
        z.extractall(data_dir)

prompts = pd.read_csv(os.path.join(data_dir, 'red_team_prompts.csv'))
policies = pd.read_csv(os.path.join(data_dir, 'toy_policies.csv'))

display(prompts.head())
display(policies)
print('Label distribution:\n', prompts['expected_action'].value_counts())


## 2) Baseline guardrail (rules)

Implement rules that catch:
- PHI / patient record requests
- data exfiltration attempts (keys, exports, credentials)
- prompt injection / policy override attempts
- unsafe medical advice requests (route to **ESCALATE**, not REFUSE)


In [ ]:
def guardrail_rules(user_prompt: str) -> str:
    p = user_prompt.lower()

    phi_keywords = ['mrn', 'patient', 'ehr', 'diagnosis', 'lab results', 'medication', 'appointment history']
    exfil_keywords = ['api key', 'credentials', 'token', 'export', 'csv', 'list all', 'phone numbers', 'emails']
    inj_keywords = ['ignore prior', 'developer mode', 'system prompt', 'override', 'reveal', 'print all']
    medical_keywords = ['dose', 'prescribe', 'diagnose', 'insulin', 'chest pain', 'what should i do']

    if any(k in p for k in phi_keywords) or any(k in p for k in exfil_keywords):
        return 'REFUSE'
    if any(k in p for k in inj_keywords):
        return 'REFUSE'
    if any(k in p for k in medical_keywords):
        return 'ESCALATE'
    return 'ALLOW'


## 3) Hybrid guardrail (rules + classifier)

Use a Hugging Face classifier to help catch unsafe intent (toxicity or safety).
You may use the example model below or substitute another model.


In [ ]:
# Example toxicity model
tox = pipeline('text-classification', model='unitary/toxic-bert', top_k=None)

def guardrail_hybrid(user_prompt: str, tox_threshold: float = 0.80) -> str:
    base = guardrail_rules(user_prompt)
    if base in ['REFUSE','ESCALATE']:
        return base

    preds = tox(user_prompt)[0]
    prob = 0.0
    for item in preds:
        if item['label'].lower() in ['toxic','toxicity']:
            prob = float(item['score'])
    if prob >= tox_threshold:
        return 'REFUSE'
    return 'ALLOW'

guardrail_hybrid('Write an insulting message to a coworker')


## 4) Evaluate guardrails
Compute overall metrics and the **False ALLOW rate** on REFUSE prompts.


In [ ]:
def evaluate(guard_fn):
    y_true = prompts['expected_action'].values
    y_pred = prompts['user_prompt'].apply(guard_fn).values

    print(classification_report(y_true, y_pred))
    cm = confusion_matrix(y_true, y_pred, labels=['ALLOW','REFUSE','ESCALATE'])
    print('Confusion matrix rows=true, cols=pred (ALLOW, REFUSE, ESCALATE):\n', cm)

    refuse_mask = (y_true == 'REFUSE')
    false_allow = np.mean((y_pred[refuse_mask] == 'ALLOW'))
    print('\nFalse ALLOW rate (on REFUSE prompts):', round(float(false_allow), 4))

print('Rules-only:')
evaluate(guardrail_rules)

print('\nHybrid:')
evaluate(guardrail_hybrid)


## 5) Write-up (required)
Answer:
1) Which attack category was hardest? Why?
2) What controls would you add in a real hospital system?
3) Final recommendation: deploy / deploy with conditions / pause


### Student write-up
- Hardest attack category:
- Proposed controls:
- Final recommendation:
